In [ ]:
from matplotlib import pyplot as plt # plotting package
import numpy as np # numpy (number + python) for working with matrices
from time import time # handy library for timing functions
import math # defines functions such as sqrt
import scipy
from IPython.utils.io import warn
from scipy.io import wavfile
from PIL import Image

from IPython.display import Audio


## Import an audio file

Before running the code below, click on the folder icon on the menu bar on the right side of the screen to open the files menu. Then upload your selected song file. Make sure to change the filename in the code below as needed.

In [ ]:
FILENAME = "one_summers_day.wav"

def read_wav(fname):
  try:
    sample_rate, data = wavfile.read(fname)
    return sample_rate, data / (2 ** 15)
  except FileNotFoundError:
    print(f'Error: The file `{fname}` was not found.')

# Convert stereo channels into a single mono channel
def to_mono(data):
  if data.ndim == 1:
    return data
  else:
    return np.mean(data, axis=1)

fs, data = read_wav(FILENAME)
a = data
data = to_mono(data)
print(data.shape)

In [ ]:





# def to_mono(data):
#   if data.ndim == 1:
#     return data
#   else:
#     return np.mean(data, axis=1)




# import numpy as np
# from pydub import AudioSegment

# FILENAME = "one_summers_day.mp3"

# def read_mp3(fname):
#     try:
#         audio = AudioSegment.from_mp3(fname)

#         sample_rate = audio.frame_rate
#         channels = audio.channels
#         sample_width = audio.sample_width  # bytes per sample

#         samples = np.array(audio.get_array_of_samples())

#         # Reshape for multi-channel audio
#         if channels > 1:
#             samples = samples.reshape((-1, channels))
#             samples = samples.mean(axis=1)  # proper mono conversion

#         # Normalize to [-1, 1]
#         max_val = float(2 ** (8 * sample_width - 1))
#         samples = samples / max_val

#         return sample_rate, samples.astype(np.float32)

#     except FileNotFoundError:
#         raise FileNotFoundError(f"Error: The file `{fname}` was not found.")

# fs, data = read_mp3(FILENAME)
# data = to_mono(data)



## Represent frequency content with Fourier Transform

This code will help you see the frequency content of your song so you can select reasonable cutoff frequencies.

In [ ]:
# Define some handy functions that abstract away some of the
# mathematical details of the Fourier transform

def time_to_freq(y, sampling_rate):
  # Returns the fourier transform of a signal
  # as well as the corresponding frequencies
  # In the context of dictionary learning, this is returning the sparse representation matrix A
  n = len(y)
  # Take the Fourier transform
  Y_full = np.fft.fft(y)
  freq_full = np.fft.fftfreq(n, d=1/sampling_rate)
  return Y_full / max(Y_full), freq_full

def plot_freq(Y, freqs, bins_per_decade=30, min_freq=20, label=""):
  # Plot only the magnitude of the FT for only positive frequencies
  # (abstract away negative frequencies and complex numbers)
  inds = freqs >= 1
  freqs = freqs[inds]
  Y = Y[inds]

  # Convert to power
  power = np.abs(Y) ** 2

  # Create log-spaced frequency bins
  f_min = max(freqs.min(), min_freq)
  f_max = freqs.max()
  num_bins = int(np.log10(f_max / f_min) * bins_per_decade)

  bin_edges = np.logspace(np.log10(f_min), np.log10(f_max), num_bins + 1)

  # Bin the power
  binned_power = np.zeros(num_bins)
  binned_freqs = np.zeros(num_bins)

  # Take the mean of the bins
  for i in range(num_bins):
      mask = (freqs >= bin_edges[i]) & (freqs < bin_edges[i + 1])
      if np.any(mask):
          binned_power[i] = np.mean(power[mask])
          binned_freqs[i] = np.sqrt(bin_edges[i] * bin_edges[i + 1])
      else:
          binned_power[i] = np.nan
          binned_freqs[i] = np.sqrt(bin_edges[i] * bin_edges[i + 1])


  plt.plot(binned_freqs, np.sqrt(binned_power), label=label)
  plt.xscale("log") # Set the x-axis to a logarithmic scale
  plt.yscale("log") # Set the y-axis to a logarhowithmic scale
  plt.xlabel("Frequency")
  plt.ylabel("Fourier transform magnitude (normalized)")

In [ ]:
# determine the maximum value for the x axis in frequency plots
MAX_FREQ_FOR_PLOTTING = 6000

In [ ]:
# plot the frequency content of the song
S, f = time_to_freq(data, fs)
plot_freq(S[f < MAX_FREQ_FOR_PLOTTING], f[f < MAX_FREQ_FOR_PLOTTING])
plt.show()

## Zoom into a specific time duration

You may want to consider the frequency content of small pieces of the song over time.  

In [ ]:
def zoom_time(y, sampling_rate, begin_s, end_s):
  return y[begin_s*sampling_rate:end_s*sampling_rate]

In [ ]:
# clip the time of the song from t_start to t_end (in seconds)
t_start = 0
t_end =400

data_clipped = zoom_time(data, fs, t_start, t_end)

In [ ]:
# listen to the audio clip
Audio(data_clipped.T, rate = fs)

In [ ]:
# visualize the frequencies in the audio clip
LOW_FREQUENCY_CUTOFF = 100  # TODO: you should change this
HIGH_FREQUENCY_CUTOFF = 700 # TODO: you should change this
S, f = time_to_freq(data_clipped, fs)
plot_freq(S[f < MAX_FREQ_FOR_PLOTTING], f[f < MAX_FREQ_FOR_PLOTTING])
plt.axvline(x=LOW_FREQUENCY_CUTOFF, color='r', linestyle='--')
plt.axvline(x=HIGH_FREQUENCY_CUTOFF, color='r', linestyle='--')
plt.show()

## Filter the signal

Once you have candidate values for the cutoff frequencies, you can apply them to your song and listen to the expected output of the filters. Use this to make sure your cutoff frequencies are reasonable - you may want to iterate! Recall your goal is that the red and green LEDs light up at different times.

In [ ]:
def apply_filter(y, sampling_rate, filter_type, cutoff_freq):
  filter_order = 2
  sos = scipy.signal.butter(filter_order, cutoff_freq, btype=filter_type,
                            fs=sampling_rate, output='sos')
  return scipy.signal.sosfilt(sos, y)

s_low = apply_filter(data, fs, 'low', cutoff_freq=LOW_FREQUENCY_CUTOFF)
s_high = apply_filter(data, fs, 'high', cutoff_freq=HIGH_FREQUENCY_CUTOFF)

In [ ]:
S_low, f_low = time_to_freq(s_low, fs)
S_high, f_high = time_to_freq(s_high, fs)
plot_freq(S[f < MAX_FREQ_FOR_PLOTTING], f[f < MAX_FREQ_FOR_PLOTTING], label="Original")
plot_freq(S_low[f_low < MAX_FREQ_FOR_PLOTTING],
          f_low[f_low < MAX_FREQ_FOR_PLOTTING], label="Low pass filtered")
plot_freq(S_high[f_high < MAX_FREQ_FOR_PLOTTING],
          f_high[f_high < MAX_FREQ_FOR_PLOTTING], label="High pass filtered")
plt.axvline(x=LOW_FREQUENCY_CUTOFF, color='r', linestyle='--')
plt.axvline(x=HIGH_FREQUENCY_CUTOFF, color='r', linestyle='--')
plt.legend()
plt.show()




In [ ]:
Audio(a.T, rate = fs)


In [ ]:
Audio(data, rate = fs)


In [ ]:
Audio(s_low.T, rate = fs)

In [ ]:
Audio(s_high.T, rate = fs)